In [14]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [15]:
df = pd.read_excel("Covid_Data_new.xlsx")

In [16]:
df.head()

,age,body_temperature,chronic_disease,breathing_issue,Blood O2 Level in Percentage,Needed Hospitalization
0,10.0,Normal,no,no,97.0,No
1,12.0,Normal,no,no,97.0,No
2,15.0,Normal,no,no,94.0,No
3,10.0,Normal,no,no,97.0,No
4,13.0,Moderate,no,no,94.0,No


In [17]:
df.isnull().sum()

age                             1
body_temperature                0
chronic_disease                 0
breathing_issue                 0
Blood O2 Level in Percentage    1
Needed Hospitalization          0
dtype: int64

In [19]:
df = df.fillna(df.mean(numeric_only=True))

In [20]:
df.isnull().sum()

age                             0
body_temperature                0
chronic_disease                 0
breathing_issue                 0
Blood O2 Level in Percentage    0
Needed Hospitalization          0
dtype: int64

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70 entries, 0 to 69
Data columns (total 6 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   age                           70 non-null     float64
 1   body_temperature              70 non-null     object 
 2   chronic_disease               70 non-null     object 
 3   breathing_issue               70 non-null     object 
 4   Blood O2 Level in Percentage  70 non-null     float64
 5   Needed Hospitalization        70 non-null     object 
dtypes: float64(2), object(4)
memory usage: 3.4+ KB


In [23]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["chronic_disease"] = le.fit_transform(df["chronic_disease"])
df["breathing_issue"] = le.fit_transform(df["breathing_issue"])
df["Needed Hospitalization"] = le.fit_transform(df["Needed Hospitalization"])

In [24]:
df.head()

,age,body_temperature,chronic_disease,breathing_issue,Blood O2 Level in Percentage,Needed Hospitalization
0,10.0,Normal,0,0,97.0,0
1,12.0,Normal,0,0,97.0,0
2,15.0,Normal,0,0,94.0,0
3,10.0,Normal,0,0,97.0,0
4,13.0,Moderate,0,0,94.0,0


In [31]:
from sklearn.preprocessing import OneHotEncoder
cls = ["body_temperature"]
ohe = OneHotEncoder(sparse_output = False,handle_unknown = "ignore")
encoded = ohe.fit_transform(df[cls])
encoded_df = pd.DataFrame(encoded,columns=ohe.get_feature_names_out(cls), index=df.index)

In [32]:
encoded_df.head()

,body_temperature_High,body_temperature_Moderate,body_temperature_Normal
0,0.0,0.0,1.0
1,0.0,0.0,1.0
2,0.0,0.0,1.0
3,0.0,0.0,1.0
4,0.0,1.0,0.0


In [34]:
df = pd.concat([df.drop(columns = cls),encoded_df],axis=1)
df.head()

,age,chronic_disease,breathing_issue,Blood O2 Level in Percentage,Needed Hospitalization,body_temperature_High,body_temperature_Moderate,body_temperature_Normal
0,10.0,0,0,97.0,0,0.0,0.0,1.0
1,12.0,0,0,97.0,0,0.0,0.0,1.0
2,15.0,0,0,94.0,0,0.0,0.0,1.0
3,10.0,0,0,97.0,0,0.0,0.0,1.0
4,13.0,0,0,94.0,0,0.0,1.0,0.0


In [35]:
X = df.drop("Needed Hospitalization",axis=1)
y = df["Needed Hospitalization"]


In [36]:
X.head()

,age,chronic_disease,breathing_issue,Blood O2 Level in Percentage,body_temperature_High,body_temperature_Moderate,body_temperature_Normal
0,10.0,0,0,97.0,0.0,0.0,1.0
1,12.0,0,0,97.0,0.0,0.0,1.0
2,15.0,0,0,94.0,0.0,0.0,1.0
3,10.0,0,0,97.0,0.0,0.0,1.0
4,13.0,0,0,94.0,0.0,1.0,0.0


In [37]:
y.head()

0    0
1    0
2    0
3    0
4    0
Name: Needed Hospitalization, dtype: int64

In [39]:
y.shape

(70,)

In [53]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.5,random_state = 42)

In [54]:
X_train.head()

,age,chronic_disease,breathing_issue,Blood O2 Level in Percentage,body_temperature_High,body_temperature_Moderate,body_temperature_Normal
50,69.0,0,1,53.0,1.0,0.0,0.0
3,10.0,0,0,97.0,0.0,0.0,1.0
17,26.0,0,0,94.0,1.0,0.0,0.0
8,18.0,0,0,66.0,0.0,1.0,0.0
55,62.0,1,1,69.0,1.0,0.0,0.0


In [55]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_s = scaler.fit_transform(X_train)
Xt_s = scaler.transform(X_test)

In [56]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression()
lr.fit(X_s,y_train)
y_pred = lr.predict(Xt_s)

In [57]:
from sklearn.metrics import confusion_matrix,accuracy_score,precision_score,recall_score,f1_score
print("Logistic Regression")
print("Precision: ",precision_score(y_test,y_pred))
print("Recall: ",recall_score(y_test,y_pred))
print("F1 score: ",f1_score(y_test,y_pred))
print("Accuracy: ",accuracy_score(y_test,y_pred))
print("CM: ",confusion_matrix(y_test,y_pred))

Logistic Regression
Precision:  0.8333333333333334
Recall:  1.0
F1 score:  0.9090909090909091
Accuracy:  0.9142857142857143
CM:  [[17  3]
 [ 0 15]]


In [58]:
from sklearn.neighbors import KNeighborsClassifier
knn_model = KNeighborsClassifier(n_neighbors=11)
knn_model.fit(X_s,y_train)
y_pred = knn_model.predict(Xt_s)

print("KNN")
print("Precision: ",precision_score(y_test,y_pred))
print("Recall: ",recall_score(y_test,y_pred))
print("F1 score: ",f1_score(y_test,y_pred))
print("Accuracy: ",accuracy_score(y_test,y_pred))
print("CM: ",confusion_matrix(y_test,y_pred))

KNN
Precision:  0.8823529411764706
Recall:  1.0
F1 score:  0.9375
Accuracy:  0.9428571428571428
CM:  [[18  2]
 [ 0 15]]


In [59]:
from sklearn.naive_bayes import GaussianNB
gnb_model = GaussianNB()
gnb_model.fit(X_s,y_train)
y_pred = gnb_model.predict(Xt_s)

print("Navie Bayes")
print("Precision: ",precision_score(y_test,y_pred))
print("Recall: ",recall_score(y_test,y_pred))
print("F1 score: ",f1_score(y_test,y_pred))
print("Accuracy: ",accuracy_score(y_test,y_pred))
print("CM: ",confusion_matrix(y_test,y_pred))

Navie Bayes
Precision:  0.7894736842105263
Recall:  1.0
F1 score:  0.8823529411764706
Accuracy:  0.8857142857142857
CM:  [[16  4]
 [ 0 15]]
